In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nyc_taxi_project.gold_data")
print("Schema created!")

In [0]:
from pyspark.sql.functions import (
    col, count, avg, sum as spark_sum, 
    round as spark_round, when
)

df_silver = spark.read.table("nyc_taxi_project.silver_data.silver_taxi")
print("Silver rows:", df_silver.count())

In [0]:
df_gold_hourly = df_silver \
    .groupBy("pickup_year", "pickup_month", "pickup_day", "pickup_hour") \
    .agg(
        count("*")                                 .alias("total_trips"),
        spark_round(avg("trip_distance"), 2)       .alias("avg_distance_miles"),
        spark_round(avg("fare_amount"), 2)         .alias("avg_fare_usd"),
        spark_round(spark_sum("fare_amount"), 2)   .alias("total_revenue_usd"),
        spark_round(avg("tip_amount"), 2)          .alias("avg_tip_usd"),
        spark_round(avg("trip_duration_minutes"),2).alias("avg_duration_mins"),
        spark_round(avg("passenger_count"), 2)     .alias("avg_passengers")
    ) \
    .orderBy("pickup_year", "pickup_month", "pickup_day", "pickup_hour")

df_gold_hourly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_project.gold_data.gold_hourly_trips")

print("gold_hourly_trips saved!")
display(df_gold_hourly.limit(10))

In [0]:
df_gold_payment = df_silver \
    .withColumn("payment_label",
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Unknown")
    ) \
    .groupBy("pickup_year", "pickup_month", "payment_label") \
    .agg(
        count("*")                              .alias("total_trips"),
        spark_round(spark_sum("fare_amount"), 2).alias("total_fare_usd"),
        spark_round(spark_sum("tip_amount"), 2) .alias("total_tips_usd"),
        spark_round(avg("fare_amount"), 2)      .alias("avg_fare_usd")
    ) \
    .orderBy("pickup_year", "pickup_month", "payment_label")

df_gold_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_project.gold_data.gold_payment_summary")

print("gold_payment_summary saved!")
display(df_gold_payment)

In [0]:
df_gold_distance = df_silver \
    .withColumn("distance_bucket",
        when(col("trip_distance") < 1,  "Short (< 1 mile)")
        .when(col("trip_distance") < 3,  "Medium (1-3 miles)")
        .when(col("trip_distance") < 10, "Long (3-10 miles)")
        .otherwise("Very Long (10+ miles)")
    ) \
    .groupBy("pickup_year", "pickup_month", "distance_bucket") \
    .agg(
        count("*")                                 .alias("total_trips"),
        spark_round(avg("fare_amount"), 2)         .alias("avg_fare_usd"),
        spark_round(avg("tip_amount"), 2)          .alias("avg_tip_usd"),
        spark_round(avg("trip_duration_minutes"),2).alias("avg_duration_mins")
    ) \
    .orderBy("pickup_year", "pickup_month", "distance_bucket")

df_gold_distance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_project.gold_data.gold_trip_distance_buckets")

print("gold_trip_distance_buckets saved!")
display(df_gold_distance)

In [0]:
spark.sql("SHOW TABLES IN nyc_taxi_project.gold_data").show()

In [0]:
spark.sql("SELECT COUNT(*) FROM nyc_taxi_project.gold_data.gold_hourly_trips").show()
spark.sql("SELECT COUNT(*) FROM nyc_taxi_project.gold_data.gold_payment_summary").show()
spark.sql("SELECT COUNT(*) FROM nyc_taxi_project.gold_data.gold_trip_distance_buckets").show()